In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Hyperparameters
batch_size = 4     # num independent examples
block_size = 1024  # max sequence length
n_embd = 768       # total embedding dim, both in and out, divisible by n_head
n_head = 12        # number of heads
assert n_embd % n_head == 0
head_size = n_embd // n_head

# Init
torch.manual_seed(42)
c_attn_W = torch.randn(2304, 768) / 2304**0.5
c_attn_b = torch.randn(2304)
c_proj_W = torch.randn(768, 768) / 768**0.5
c_proj_b = torch.randn(768)
x = torch.randn(batch_size,block_size,n_embd)

In [3]:
# Reference Implementation

class CausalSelfAttentionMarcin(nn.Module):
    """Multiple self-attention heads"""
    def __init__(self, n_head, n_embd):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head

        self.c_attn = nn.Linear(n_embd, 3*n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1  # flag to scale proj into residual

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)  # B, T, nh*hs
        q = q.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        k = k.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        v = v.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        q = q.transpose(1, 2)  # B,nh,T,hs
        k = k.transpose(1, 2)  # B,nh,T,hs
        v = v.transpose(1, 2)  # B,nh,T,hs

        # W_affin = q @ k.mT / k.shape[-1]**0.5  # B,nh,T,hs @ B,nh,hs,T -> B,nh,T,T
        # W_affin = W_affin.masked_fill(self.bias[:,:,:T,:T]==0, float('-inf'))
        # W_affin = torch.softmax(W_affin, dim=-1)  # B,nh,T,T
        # y = W_affin @ v    # B,nh,T,T @ B,nh,T,hs -> B,nh,T,hs
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        y = y.transpose(1, 2)  # B,T,nh,hs
        y = y.contiguous()
        y = y.view(B,T,C)

        out = self.c_proj(y)
        return out

In [4]:
# Run reference implementation

csa_m = CausalSelfAttentionMarcin(n_head=n_head, n_embd=n_embd)
csa_m_state = csa_m.state_dict()
csa_m_state['c_attn.weight'] = c_attn_W.clone()
csa_m_state['c_attn.bias'] = c_attn_b.clone()
csa_m_state['c_proj.weight'] = c_proj_W.clone()
csa_m_state['c_proj.bias'] = c_proj_b.clone()
csa_m.load_state_dict(csa_m_state)

y_m2 = csa_m(x)
print(y_m2.shape)
print(y_m2.sum().item())

torch.Size([4, 1024, 768])
-18783.599609375


In [5]:
# RoPE Implementation (no class)

# Initialize linear projections
c_q = nn.Linear(n_embd, n_embd)
c_k = nn.Linear(n_embd, n_embd)
c_v = nn.Linear(n_embd, n_embd)
c_proj = nn.Linear(n_embd, n_embd)

with torch.no_grad():
    c_q.weight.copy_(c_attn_W[:n_embd])
    c_q.bias.copy_(c_attn_b[:n_embd])
    c_k.weight.copy_(c_attn_W[n_embd:2*n_embd])
    c_k.bias.copy_(c_attn_b[n_embd:2*n_embd])
    c_v.weight.copy_(c_attn_W[2*n_embd:])
    c_v.bias.copy_(c_attn_b[2*n_embd:])
    c_proj.weight.copy_(c_proj_W)
    c_proj.bias.copy_(c_proj_b)
    

In [32]:
# Compute exponent for the RoPE frequencies
theta = torch.arange(0, head_size, step=2)
theta = theta/head_size    # no 2* because step=2 already
theta = 10_000**-theta
print(theta.shape)
print(theta[:6])
print(theta[-4:])


torch.Size([32])
tensor([1.0000, 0.7499, 0.5623, 0.4217, 0.3162, 0.2371])
tensor([0.0003, 0.0002, 0.0002, 0.0001])


In [33]:
# Positions
pos = torch.arange(0, 10*1024)  # overprovision to support longer sequences
pos.shape

torch.Size([10240])

In [34]:
# Multiply out
tmp = torch.outer(pos, theta)
print(tmp.shape)
tmp[:4, :8]

torch.Size([10240, 32])


tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.7499, 0.5623, 0.4217, 0.3162, 0.2371, 0.1778, 0.1334],
        [2.0000, 1.4998, 1.1247, 0.8434, 0.6325, 0.4743, 0.3557, 0.2667],
        [3.0000, 2.2497, 1.6870, 1.2651, 0.9487, 0.7114, 0.5335, 0.4001]])

In [35]:
# Compute sin and cos
sin = torch.sin(tmp)
cos = torch.cos(tmp)
print(sin.shape, cos.shape)
print(sin[:4, :8])
print(cos[:4, :8])

torch.Size([10240, 32]) torch.Size([10240, 32])
tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8415, 0.6816, 0.5332, 0.4093, 0.3110, 0.2349, 0.1769, 0.1330],
        [0.9093, 0.9975, 0.9021, 0.7469, 0.5911, 0.4567, 0.3482, 0.2636],
        [0.1411, 0.7783, 0.9933, 0.9536, 0.8126, 0.6529, 0.5085, 0.3895]])
tensor([[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.7318,  0.8460,  0.9124,  0.9504,  0.9720,  0.9842,  0.9911],
        [-0.4161,  0.0709,  0.4315,  0.6649,  0.8066,  0.8896,  0.9374,  0.9646],
        [-0.9900, -0.6279, -0.1160,  0.3010,  0.5828,  0.7574,  0.8610,  0.9210]])


In [36]:
x[:2,:2,:4]

tensor([[[ 0.3200, -0.2651, -0.0264,  2.1537],
         [ 2.2608,  1.7447,  0.7011,  0.3407]],

        [[ 0.8515, -0.0897,  0.1958, -1.5051],
         [-1.2797, -1.5142, -0.0516,  0.5504]]])

In [37]:
B, T, C = x.size()

In [39]:
# Trim sin, cos to T and add batch dim
sin_trimmed = sin[:T, :].view(1, T, 1, head_size//2)
cos_trimmed = cos[:T, :].view(1, T, 1, head_size//2)
print(sin_trimmed.shape)
print(cos_trimmed.shape)

torch.Size([1, 1024, 1, 32])
torch.Size([1, 1024, 1, 32])


In [40]:
# Calculate outputs with RoPE
with torch.no_grad():
    q = c_q(x)    # B, T, nh*hs
    k = c_k(x)    # B, T, nh*hs
    v = c_v(x)    # B, T, nh*hs
    q = q.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    k = k.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    v = v.view(B, T, n_head, C//n_head)  # B,T,nh,hs



In [41]:
q.shape, k.shape, v.shape

(torch.Size([4, 1024, 12, 64]),
 torch.Size([4, 1024, 12, 64]),
 torch.Size([4, 1024, 12, 64]))

In [51]:
print(q[0, 1, 0, -4:])
q_x = q[..., :head_size//2]
q_y = q[..., head_size//2:]
print(q_x.shape)
print(q_y.shape)
q_x_rot = cos_trimmed * q_x - sin_trimmed * q_y
q_y_rot = sin_trimmed * q_x + cos_trimmed * q_y
q_rot = torch.cat([q_x_rot, q_y_rot], dim=-1)
print(q_rot.shape)
print(q_rot[0, 1, 0, -4:])

tensor([-1.5068,  1.4503, -0.3877,  1.7119])
torch.Size([4, 1024, 12, 32])
torch.Size([4, 1024, 12, 32])
torch.Size([4, 1024, 12, 64])
tensor([-1.5075,  1.4499, -0.3878,  1.7120])


In [52]:
print(k[0, 1, 0, -4:])
k_x = k[..., :head_size//2]
k_y = k[..., head_size//2:]
print(k_x.shape)
print(k_y.shape)
k_x_rot = cos_trimmed * k_x - sin_trimmed * k_y
k_y_rot = sin_trimmed * k_x + cos_trimmed * k_y
k_rot = torch.cat([k_x_rot, k_y_rot], dim=-1)
print(k_rot.shape)
print(k_rot[0, 1, 0, -4:])

tensor([ 0.2733,  1.1050, -2.8900, -0.7209])
torch.Size([4, 1024, 12, 32])
torch.Size([4, 1024, 12, 32])
torch.Size([4, 1024, 12, 64])
tensor([ 0.2728,  1.1050, -2.8901, -0.7211])


In [53]:
with torch.no_grad():
    q = q_rot.transpose(1, 2)  # B,nh,T,hs
    k = k_rot.transpose(1, 2)  # B,nh,T,hs
    v = v.transpose(1, 2)  # B,nh,T,hs

    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    y = y.transpose(1, 2)  # B,T,nh,hs
    y = y.contiguous()
    y = y.view(B,T,C)

    out = c_proj(y)

    print(out.sum().item())

-21446.939453125


In [ ]:
# Itnore rest of notebook
# --- IGNORE ---

In [ ]:
class CausalSelfAttentionMarcin2(nn.Module):
    """Multiple self-attention heads"""
    def __init__(self, n_head, n_embd):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head

        self.c_q = nn.Linear(n_embd, n_embd)
        self.c_k = nn.Linear(n_embd, n_embd)
        self.c_v = nn.Linear(n_embd, n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)


    def forward(self, x):
        B, T, C = x.size()

        q = self.c_q(x)    # B, T, nh*hs
        k = self.c_k(x)    # B, T, nh*hs
        v = self.c_v(x)    # B, T, nh*hs

        q = q.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        k = k.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        v = v.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        q = q.transpose(1, 2)  # B,nh,T,hs
        k = k.transpose(1, 2)  # B,nh,T,hs
        v = v.transpose(1, 2)  # B,nh,T,hs

        # W_affin = q @ k.mT / k.shape[-1]**0.5  # B,nh,T,hs @ B,nh,hs,T -> B,nh,T,T
        # W_affin = W_affin.masked_fill(self.bias[:,:,:T,:T]==0, float('-inf'))
        # W_affin = torch.softmax(W_affin, dim=-1)  # B,nh,T,T
        # y = W_affin @ v    # B,nh,T,T @ B,nh,T,hs -> B,nh,T,hs
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        y = y.transpose(1, 2)  # B,T,nh,hs
        y = y.contiguous()
        y = y.view(B,T,C)

        out = self.c_proj(y)
        return out

In [ ]:
with torch.no_grad():
    B, T, C = x.size()

    q = c_q(x)    # B, T, nh*hs
    k = c_k(x)    # B, T, nh*hs
    v = c_v(x)    # B, T, nh*hs
    q = q.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    k = k.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    v = v.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    q = q.transpose(1, 2)  # B,nh,T,hs
    k = k.transpose(1, 2)  # B,nh,T,hs
    v = v.transpose(1, 2)  # B,nh,T,hs

    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    y = y.transpose(1, 2)  # B,T,nh,hs
    y = y.contiguous()
    y = y.view(B,T,C)

    out = c_proj(y)

    print(out.sum().item())


In [ ]:
csa_m2 = CausalSelfAttentionMarcin2(n_head=n_head, n_embd=n_embd)
csa_m2_state = csa_m2.state_dict()

csa_m2_state['c_q.weight'] = c_attn_W[:n_embd].clone()
csa_m2_state['c_q.bias'] = c_attn_b[:n_embd].clone()
csa_m2_state['c_k.weight'] = c_attn_W[n_embd:2*n_embd].clone()
csa_m2_state['c_k.bias'] = c_attn_b[n_embd:2*n_embd].clone()
csa_m2_state['c_v.weight'] = c_attn_W[2*n_embd:].clone()
csa_m2_state['c_v.bias'] = c_attn_b[2*n_embd:].clone()

csa_m2_state['c_proj.weight'] = c_proj_W.clone()
csa_m2_state['c_proj.bias'] = c_proj_b.clone()
csa_m2.load_state_dict(csa_m2_state)

y_m2 = csa_m2(x)
print(y_m2.shape)
print(y_m2.sum().item())